# struct-extract-eval: End-to-End Demo

This notebook walks through the full evaluation workflow:

1. **Create data** -- synthetic materials science gold + extracted pairs
2. **Infer schema** -- derive a resolved schema from gold instances
3. **Annotate** -- add default `x-eval-*` comparators to resolved schema to get eval schema
4. **Edit** -- simulate user adjustments to the eval schema
5. **Evaluate** -- run the pipeline
6. **Inspect** -- view results, per-field breakdown, worst records

## Helpers: diff display

In [22]:
import json
import difflib
from copy import deepcopy
from html import escape
from IPython.display import HTML, display


def show_json(data: dict, title: str = "") -> None:
    """Display a JSON object with syntax highlighting."""
    text = escape(json.dumps(data, indent=2, ensure_ascii=False))
    header = f"<h4>{escape(title)}</h4>" if title else ""
    display(HTML(f"""{header}<pre style="background:#f6f8fa; padding:12px; border-radius:6px;
        border:1px solid #d0d7de; font-size:13px; line-height:1.5;
        max-height:500px; overflow:auto;">{text}</pre>"""))


def show_diff(before: dict, after: dict, before_title: str = "Before", after_title: str = "After") -> None:
    """Show a unified diff of two JSON objects, color-coded like git diff."""
    before_lines = json.dumps(before, indent=2, ensure_ascii=False).splitlines()
    after_lines = json.dumps(after, indent=2, ensure_ascii=False).splitlines()

    diff = list(difflib.unified_diff(before_lines, after_lines, lineterm="",
                                      fromfile=before_title, tofile=after_title, n=3))
    if not diff:
        display(HTML("<p><em>No differences</em></p>"))
        return

    html_lines = []
    for line in diff:
        escaped = escape(line)
        if line.startswith("+++") or line.startswith("---"):
            html_lines.append(f'<span style="color:#656d76; font-weight:bold;">{escaped}</span>')
        elif line.startswith("@@"):
            html_lines.append(f'<span style="color:#8250df;">{escaped}</span>')
        elif line.startswith("+"):
            html_lines.append(f'<span style="background:#dafbe1; color:#1a7f37;">{escaped}</span>')
        elif line.startswith("-"):
            html_lines.append(f'<span style="background:#ffebe9; color:#cf222e;">{escaped}</span>')
        else:
            html_lines.append(escaped)

    joined = "\n".join(html_lines)
    display(HTML(f"""<pre style="background:#f6f8fa; padding:12px; border-radius:6px;
        border:1px solid #d0d7de; font-size:13px; line-height:1.6;
        max-height:600px; overflow:auto;">{joined}</pre>"""))


def show_pair(gold: dict, extracted: dict, record_id: str | int = "") -> None:
    """Show gold vs extracted side by side."""
    gold_text = escape(json.dumps(gold, indent=2, ensure_ascii=False))
    ext_text = escape(json.dumps(extracted, indent=2, ensure_ascii=False))
    title = f"<h4>Record {escape(str(record_id))}</h4>" if record_id != "" else ""
    display(HTML(f"""{title}
    <div style="display:flex; gap:16px;">
        <div style="flex:1;">
            <div style="font-weight:bold; margin-bottom:4px; color:#656d76;">Gold</div>
            <pre style="background:#f6f8fa; padding:12px; border-radius:6px;
                border:1px solid #d0d7de; font-size:13px; line-height:1.5;
                max-height:400px; overflow:auto;">{gold_text}</pre>
        </div>
        <div style="flex:1;">
            <div style="font-weight:bold; margin-bottom:4px; color:#656d76;">Extracted</div>
            <pre style="background:#f6f8fa; padding:12px; border-radius:6px;
                border:1px solid #d0d7de; font-size:13px; line-height:1.5;
                max-height:400px; overflow:auto;">{ext_text}</pre>
        </div>
    </div>"""))


def show_table(headers: list[str], rows: list[list]) -> None:
    """Render a table with optional color coding for status columns."""
    status_colors = {
        "match": "#1a7f37", "mismatch": "#cf222e",
        "omission": "#9a6700", "hallucination": "#8250df", "skipped": "#656d76",
    }
    th = "".join(
        f"<th style='text-align:left; padding:6px 12px; border-bottom:2px solid #d0d7de;'>{escape(str(h))}</th>"
        for h in headers
    )
    trs = []
    for row in rows:
        tds = []
        for val in row:
            val_str = escape(str(val))
            color = status_colors.get(str(val), "")
            style = f"color:{color}; font-weight:bold;" if color else ""
            tds.append(f"<td style='padding:4px 12px; border-bottom:1px solid #eaeef2; {style}'>{val_str}</td>")
        trs.append(f"<tr>{''.join(tds)}</tr>")
    tbody = "".join(trs)
    display(HTML(f"""<table style="border-collapse:collapse; font-size:13px; font-family:monospace;">
        <thead><tr>{th}</tr></thead><tbody>{tbody}</tbody></table>"""))

## 1. Synthetic Data

3 records of materials science extraction covering the key scenarios:

1. **Synonym + hallucination** -- method synonym (CVD), extra hallucinated step
2. **Omission + wrong value** -- missing step, wrong temperature, optional field missing
3. **Perfect match** -- everything correct, including an optional field absent in both

In [23]:
GOLD = [
    {
        # Record 0: has lab_id, two steps
        "method": "Chemical Vapor Deposition",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
        ],
    },
    {
        # Record 1: has lab_id, two steps
        "method": "Sputtering",
        "temperature": 500.0,
        "lab_id": "LAB-002",
        "steps": [
            {"name": "clean", "duration": 30},
            {"name": "deposit", "duration": 90},
        ],
    },
    {
        # Record 2: NO lab_id -- makes lab_id optional in inferred schema
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

EXTRACTED = [
    {
        # Record 0: method synonym (CVD), extra hallucinated step
        "method": "CVD",
        "temperature": 773.15,
        "lab_id": "LAB-001",
        "steps": [
            {"name": "deposit", "duration": 120},
            {"name": "anneal", "duration": 60},
            {"name": "cool", "duration": 30},  # hallucinated
        ],
    },
    {
        # Record 1: temp wrong (450 vs 500), missing second step, lab_id missing
        "method": "Sputtering",
        "temperature": 450.0,
        "steps": [
            {"name": "clean", "duration": 30},
        ],
    },
    {
        # Record 2: all correct, no lab_id in gold either
        "method": "Pulsed Laser Deposition",
        "temperature": 700.0,
        "steps": [
            {"name": "ablate", "duration": 45},
            {"name": "deposit", "duration": 90},
        ],
    },
]

### Gold vs Extracted: side-by-side

Inspect a few pairs to see the kind of errors the extractor makes.

In [24]:
# Show all 3 pairs
for i in range(len(GOLD)):
    show_pair(GOLD[i], EXTRACTED[i], record_id=i)

## 2. Infer Schema from Gold Instances

`infer_schema()` examines all gold records and produces a resolved JSON Schema.
Fields absent in some instances are excluded from the `required` array.

In [25]:
from struct_extract_eval import infer_schema, add_default_xeval, evaluate

resolved_schema = infer_schema(GOLD)
show_json(resolved_schema, "Resolved Schema (inferred from gold)")

Notice: `lab_id` is **not** in the `required` array because record 2 doesn't have it.

## 3. Annotate with Eval Defaults

`add_default_xeval()` adds `x-eval-compare` to each leaf field based on its type,
and converts the `required` array into per-field `x-eval-required: false`.

In [26]:
eval_schema = deepcopy(resolved_schema)
add_default_xeval(eval_schema)

show_diff(resolved_schema, eval_schema,
          before_title="Resolved Schema",
          after_title="Eval Schema (with x-eval-* defaults)")

## 4. User Edits the Eval Schema

The defaults are a best guess. In practice you'd save the eval schema to a file,
edit it, and load it back. Here we simulate edits in code:

- **method**: `exact` -> `oneof` with known synonyms (CVD, PVD, etc.)
- **temperature**: `numeric` (default, no tolerance) -> `numeric` with 1% relative tolerance

In [27]:
before_edit = deepcopy(eval_schema)

# method: exact -> oneof with known synonyms
eval_schema["properties"]["method"]["x-eval-compare"] = {
    "oneof": {
        "values": [
            "Chemical Vapor Deposition", "CVD",
            "Sputtering", "Sputter Deposition",
            "Pulsed Laser Deposition", "PLD",
        ]
    }
}

# temperature: add 1% relative tolerance
eval_schema["properties"]["temperature"]["x-eval-compare"] = {
    "numeric": {"tolerance": {"rel": 0.01}}
}

show_diff(before_edit, eval_schema,
          before_title="Eval Schema (defaults)",
          after_title="Eval Schema (after user edits)")

## 5. Run Evaluation

In [28]:
run = evaluate(GOLD, EXTRACTED, schema=eval_schema)

## 6. Inspect Results

### Run Summary

In [29]:
show_table(
    ["Metric", "Value"],
    [
        ["Records", run.total_records],
        ["Fields scored", run.total_fields],
        ["Precision", f"{run.mean_precision:.3f}"],
        ["Recall", f"{run.mean_recall:.3f}"],
        ["F1", f"{run.mean_f1:.3f}"],
        ["Omissions (FN)", run.total_omissions],
        ["Hallucinations (FP)", run.total_hallucinations],
    ],
)

Metric,Value
Records,3
Fields scored,21
Precision,0.843
Recall,0.833
F1,0.825
Omissions (FN),2
Hallucinations (FP),2


### Per-Field Breakdown

Which fields does the extractor struggle with?

In [30]:
show_table(
    ["Field Path", "Mean Score", "Matches", "Mismatches", "Omissions", "Hallucinations"],
    [
        [path, f"{agg.mean_score:.2f}", agg.matches, agg.mismatches, agg.omissions, agg.hallucinations]
        for path, agg in sorted(run.per_field.items())
    ],
)

Field Path,Mean Score,Matches,Mismatches,Omissions,Hallucinations
lab_id,1.00,1,0,0,0
method,1.00,3,0,0,0
steps[].duration,0.71,5,0,1,1
steps[].name,0.71,5,0,1,1
temperature,0.67,2,1,0,0


### Worst Records

Drill into the records with the lowest F1 to see what went wrong.

In [31]:
worst = sorted(run.records, key=lambda r: r.f1)[:3]

for record in worst:
    display(HTML(f"<h4>Record {record.record_id} &mdash; "
                 f"F1: {record.f1:.3f} &nbsp; P: {record.precision:.3f} &nbsp; R: {record.recall:.3f}</h4>"))

    # Side-by-side gold vs extracted
    show_pair(record.gold, record.extracted)

    # Field-by-field results
    show_table(
        ["Field", "Gold", "Extracted", "Score", "Status"],
        [
            [fr.path, str(fr.gold_value)[:30], str(fr.extracted_value)[:30],
             f"{fr.score:.1f}", fr.status]
            for fr in record.field_results
        ],
    )

Field,Gold,Extracted,Score,Status
method,Sputtering,Sputtering,1.0,match
steps[].duration,30,30,1.0,match
steps[].name,clean,clean,1.0,match
steps[].duration,None,None,0.0,omission
steps[].name,None,None,0.0,omission
temperature,500.0,450.0,0.0,mismatch


Field,Gold,Extracted,Score,Status
lab_id,LAB-001,LAB-001,1.0,match
method,Chemical Vapor Deposition,CVD,1.0,match
steps[].duration,120,120,1.0,match
steps[].name,deposit,deposit,1.0,match
steps[].duration,60,60,1.0,match
steps[].name,anneal,anneal,1.0,match
steps[].duration,None,30,0.0,hallucination
steps[].name,None,cool,0.0,hallucination
temperature,773.15,773.15,1.0,match


Field,Gold,Extracted,Score,Status
method,Pulsed Laser Deposition,Pulsed Laser Deposition,1.0,match
steps[].duration,45,45,1.0,match
steps[].name,ablate,ablate,1.0,match
steps[].duration,90,90,1.0,match
steps[].name,deposit,deposit,1.0,match
temperature,700.0,700.0,1.0,match
